In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [2]:
import random
import numpy as np
import torch

def set_seed(seed: int):
    # Python
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # Numpy
    np.random.seed(seed)

    # Torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # cuDNN (중요)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # PyTorch 2.x (있으면 더 강력)
    torch.use_deterministic_algorithms(True)

seed = 0
set_seed(seed)

In [3]:
# ================================================================
# Step 1: 데이터 로더 (ETTh1 / Weather)
# ================================================================
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

class TSDataset(Dataset):
    def __init__(self, data_path, flag='train',
                 seq_len=336, pred_len=96,
                 scale=True, n_vars_limit=None):   # ← n_vars_limit 추가
        assert flag in ['train', 'val', 'test']
        self.seq_len  = seq_len
        self.pred_len = pred_len
        self.scale    = scale

        df = pd.read_csv(data_path)

        if 'date' in df.columns:
            df = df.drop(columns=['date'])

        data = df.values.astype(np.float32)

        # ← 이 블록 추가
        if n_vars_limit is not None:
            data = data[:, :n_vars_limit]

        T = len(data)

        border = {
            'train': (0,            int(T * 0.6)),
            'val':   (int(T * 0.6), int(T * 0.8)),   # 1311 → 2881개로 증가
            'test':  (int(T * 0.8), T),
        }
        s, e = border[flag]

        scaler = StandardScaler()
        scaler.fit(data[:int(T * 0.7)])
        self.data = scaler.transform(data)[s:e]

    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx):
        x = self.data[idx            : idx + self.seq_len]
        y = self.data[idx+self.seq_len : idx+self.seq_len+self.pred_len]
        return {
            "x": torch.tensor(x).T,
            "y": torch.tensor(y).T,
        }


def get_ts_loader(dataset_name, batch_size=32, seq_len=336, pred_len=96,
                  data_root='./data', n_vars_limit=None):
    """
    dataset_name: 'ETTh1' | 'Weather'
    반환: train_loader, val_loader, test_loader, n_vars
    """
    paths = {
            'ETTh1':       './data/ETT-small/ETTh1.csv',
            'ETTh2':       './data/ETT-small/ETTh2.csv',
            'ETTm1':       './data/ETT-small/ETTm1.csv',
            'Weather':     './data/weather/weather.csv',
            'Electricity': './data/electricity/electricity.csv',  # 321 vars
            'Traffic':     './data/traffic/traffic.csv',          # 862 vars
        }
    assert dataset_name in paths, f"지원하지 않는 데이터셋: {dataset_name}"
    path = paths[dataset_name]

    loaders = {}
    for flag in ['train', 'val', 'test']:
        ds = TSDataset(path, flag=flag, seq_len=seq_len, pred_len=pred_len,
                       n_vars_limit=n_vars_limit)
        loaders[flag] = DataLoader(
            ds, batch_size=batch_size,
            shuffle=(flag == 'train'),
            num_workers=2, pin_memory=True, drop_last=(flag=='train')
        )

    # 변수(채널) 수 확인
    sample = next(iter(loaders['train']))
    n_vars = sample['x'].shape[1]   # (B, C, T) → C

    print(f"[{dataset_name}] n_vars={n_vars}, "
          f"train={len(loaders['train'].dataset)}, "
          f"val={len(loaders['val'].dataset)}, "
          f"test={len(loaders['test'].dataset)}")

    return loaders['train'], loaders['val'], loaders['test'], n_vars


In [4]:
import torch
import torch.nn as nn
from tqdm.notebook import tqdm
import copy
# ================================================================
# Step 2: PatchTST / DLinear
# ================================================================

class RevIN(nn.Module):
    def __init__(self, num_features, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.affine_weight = nn.Parameter(torch.ones(1, num_features, 1))
        self.affine_bias = nn.Parameter(torch.zeros(1, num_features, 1))

    def forward(self, x, mode='norm'):
        if mode == 'norm':
            self.mean = x.mean(dim=-1, keepdim=True)
            self.stdev = torch.sqrt(torch.var(x, dim=-1, keepdim=True, unbiased=False) + self.eps)
            return (x - self.mean) / (self.stdev + self.eps) * self.affine_weight + self.affine_bias
        else: # denorm
            return (x - self.affine_bias) / (self.affine_weight + self.eps) * self.stdev + self.mean

class MLP(nn.Module):
    def __init__(self, in_features, out_features, hidden_dim=64, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_features)
        )
    def forward(self, x):
        return self.net(x)
        
# ── TimesNet ──────────────────────────────────────────────────────
class TimesNet(nn.Module):
    """
    TimesNet (ICLR 2023) 경량 구현
    1D 시계열 → 2D reshape → Conv2d → 예측
    """
    def __init__(self, seq_len, pred_len, n_vars,
                 d_model=64, n_kernels=3, top_k=3, dropout=0.1):
        super().__init__()
        self.revin   = RevIN(n_vars)
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.top_k   = top_k

        # 입력 임베딩
        self.embed = nn.Linear(1, d_model)

        # 2D Conv 블록 (period별)
        self.conv_blocks = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(d_model, d_model, kernel_size=3, padding=1),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Conv2d(d_model, d_model, kernel_size=3, padding=1),
            )
            for _ in range(n_kernels)
        ])

        # 예측 헤드
        self.head = nn.Linear(d_model * seq_len, pred_len)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """x: (B, C, seq_len)"""
        B, C, T = x.shape
        x = self.revin(x, 'norm')

        # 채널 독립: (B*C, T, 1) → embed → (B*C, T, d_model)
        xr = x.reshape(B * C, T, 1)
        z  = self.embed(xr)                    # (B*C, T, d_model)
        z  = z.permute(0, 2, 1)               # (B*C, d_model, T)

        # FFT로 top-k 주기 추출
        freq  = torch.fft.rfft(z, dim=-1)
        amp   = freq.abs().mean(dim=1)         # (B*C, T//2+1)
        _, topk_idx = torch.topk(amp, self.top_k, dim=-1)

        out = torch.zeros_like(z)
        for i, conv in enumerate(self.conv_blocks):
            # i번째 top 주기로 2D reshape
            period = (topk_idx[:, i % self.top_k] % (T // 2) + 2).clamp(2, T)
            p = period[0].item()               # 배치 내 대표값 사용
            L = ((T + p - 1) // p) * p
            # 패딩 후 2D reshape
            zp = F.pad(z, (0, L - T))         # (B*C, d_model, L)
            zp = zp.reshape(B * C, z.size(1), L // p, p)
            zp = conv(zp)                      # (B*C, d_model, L//p, p)
            zp = zp.reshape(B * C, z.size(1), L)[:, :, :T]
            out = out + zp

        out = out.reshape(B * C, -1)           # (B*C, d_model*T)
        out = self.dropout(out)
        out = self.head(out)                   # (B*C, pred_len)
        out = out.reshape(B, C, self.pred_len)
        return self.revin(out, 'denorm')

# NLinear — 단기 변동에 강함
class NLinear(nn.Module):
    def __init__(self, seq_len, pred_len, n_vars):
        super().__init__()
        self.revin    = RevIN(n_vars)
        self.seq_len  = seq_len
        self.pred_len = pred_len
        self.proj     = nn.Linear(seq_len, pred_len)

    def forward(self, x):
        """x: (B, C, seq_len)"""
        x    = self.revin(x, 'norm')
        last = x[:, :, -1:]                  # (B, C, 1)
        x_shift = x - last                   # (B, C, seq_len) — last값 제거

        # proj는 마지막 차원(seq_len)에 적용 → (B, C, pred_len)
        out = self.proj(x_shift) + self.proj(
              last.expand(-1, -1, self.seq_len)  # (B,C,1) → (B,C,seq_len)으로 확장 후 proj
        )
        return self.revin(out, 'denorm')


# ── TSMixer ─────────────────────────────────────────────────────
class MixerBlock(nn.Module):
    def __init__(self, n_vars, seq_len, dropout=0.1):
        super().__init__()
        # Time-Mixing: 시간 축을 따라 선형 결합 (T -> T)
        self.time_mixing = nn.Sequential(
            nn.Linear(seq_len, seq_len),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(seq_len)
        
        # Channel-Mixing: 채널 축을 따라 선형 결합 (C -> C)
        self.channel_mixing = nn.Sequential(
            nn.Linear(n_vars, n_vars),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.norm2 = nn.LayerNorm(n_vars)

    def forward(self, x):
        # x: (B, C, T)
        # 1. Time Mixing
        res = x
        x = self.time_mixing(x) + res
        x = self.norm1(x)
        
        # 2. Channel Mixing
        res = x
        x = x.transpose(1, 2)  # (B, T, C)로 변경하여 채널 믹싱
        x = self.channel_mixing(x) + res.transpose(1, 2)
        x = self.norm2(x).transpose(1, 2) # 다시 (B, C, T)로 복귀
        return x

class TSMixer(nn.Module):
    def __init__(self, seq_len, pred_len, n_vars, n_layers=2, dropout=0.1):
        super().__init__()
        self.revin = RevIN(n_vars)
        self.mixer_blocks = nn.ModuleList([
            MixerBlock(n_vars, seq_len, dropout) for _ in range(n_layers)
        ])
        # 최종 예측: (B, C, seq_len) -> (B, C, pred_len)
        self.head = nn.Linear(seq_len, pred_len)

    def forward(self, x):
        # 1. RevIN
        x = self.revin(x, 'norm')
        
        # 2. Mixing Layers
        for block in self.mixer_blocks:
            x = block(x)
            
        # 3. Head (Time-wise projection)
        out = self.head(x) # (B, C, pred_len)
        
        # 4. Denorm
        return self.revin(out, 'denorm')
        
# ── PatchTST ─────────────────────────────────────────────────────
class PatchTST(nn.Module):
    def __init__(self, seq_len, pred_len, n_vars, patch_len=16, stride=8,
                 d_model=128, n_heads=8, n_layers=3, d_ff=256, dropout=0.1):
        super().__init__()
        self.revin = RevIN(n_vars) # RevIN 적용
        self.patch_len = patch_len
        self.stride = stride
        self.n_patches = (seq_len - patch_len) // stride + 1

        self.patch_embed = nn.Linear(patch_len, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.n_patches, d_model))
        self.dropout = nn.Dropout(dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        # 예측 헤드: MLP로 강화
        self.head = MLP(self.n_patches * d_model, pred_len, hidden_dim=d_ff)

    def forward(self, x):
        x = self.revin(x, 'norm') # RevIN 적용
        B, C, T = x.shape
        
        # Patching
        x_patch = x.unfold(-1, self.patch_len, self.stride).reshape(B * C, self.n_patches, self.patch_len)
        z = self.patch_embed(x_patch) + self.pos_embed
        z = self.dropout(z)
        
        # Transformer
        z = self.encoder(z)
        
        # Prediction
        z = z.reshape(B * C, -1)
        out = self.head(z)
        out = out.reshape(B, C, -1)
        
        return self.revin(out, 'denorm') # 역정규화 적용

# ── 모델 팩토리 ───────────────────────────────────────────────────
def build_model(model_type, n_vars, seq_len, pred_len, device):
    if model_type == 'PatchTST':
        model = PatchTST(
            seq_len=seq_len, pred_len=pred_len, n_vars=n_vars,
            patch_len=16, stride=8,
            d_model=128, n_heads=8, n_layers=3
        )
    elif model_type == 'TimesNet':
        model = TimesNet(seq_len=seq_len, pred_len=pred_len, n_vars=n_vars)
    elif model_type == 'NLinear':
        model = NLinear(seq_len=seq_len, pred_len=pred_len, n_vars=n_vars)
    elif model_type == 'TSMixer':
        model = TSMixer(seq_len=seq_len, pred_len=pred_len, n_vars=n_vars)
    else:
        raise ValueError(f"Unknown model: {model_type}")
    return model.to(device)

In [5]:
# ================================================================
# fine_tune — 변경 없음 (이미 shape-agnostic)
# ================================================================
def fine_tune(model, optimizer, criterion, train_loader, device):
    model.train()
    train_loss = 0.0

    for batch in train_loader:
        inputs  = batch["x"].to(device)
        targets = batch["y"].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    return model, copy.deepcopy(model.state_dict()), train_loss / len(train_loader)


# ================================================================
# ssml — 핵심 수정: 4D 인덱싱 → Boolean mask
# ================================================================
def ssml(model1, model2, optimizer1, optimizer2,
                  criterion, train_loader, device,
                  margin=0.005, dist_weight=0.5):
    model1.train(); model2.train()
    train_loss1 = train_loss2 = 0.0

    for batch in train_loader:
        inputs  = batch["x"].to(device)   # (B, C, T_in)
        targets = batch["y"].to(device)   # (B, C, T_out)

        optimizer1.zero_grad(); optimizer2.zero_grad()
        out1 = model1(inputs)   # (B, C, T_out)
        out2 = model2(inputs)

        # element-wise 오차
        loss_map1 = torch.abs(out1 - targets)   # (B, C, T_out)
        loss_map2 = torch.abs(out2 - targets)

        # ── 핵심: timestep 단위로 집계 ──────────────────────────
        # 채널 평균 → (B, T_out) : 각 시점에서 평균 오차
        err1_t = loss_map1.mean(dim=1)   # (B, T_out)
        err2_t = loss_map2.mean(dim=1)

        # timestep 단위 마스킹
        imitate1_t = err2_t < (err1_t - margin)   # (B, T_out)
        imitate2_t = err1_t < (err2_t - margin)
        sup_t      = ~(imitate1_t | imitate2_t)

        # (B, T_out) → (B, C, T_out)으로 확장
        imitate1 = imitate1_t.unsqueeze(1).expand_as(out1)
        imitate2 = imitate2_t.unsqueeze(1).expand_as(out2)
        sup      = sup_t.unsqueeze(1).expand_as(out1)

        # Model1 loss
        loss1 = torch.tensor(0.0, device=device)
        if imitate1.any():
            loss1 = loss1 + dist_weight * criterion(
                out1[imitate1], out2[imitate1].detach())
        if sup.any():
            loss1 = loss1 + criterion(out1[sup], targets[sup])
        loss1.backward()
        optimizer1.step()

        # Model2 loss
        loss2 = torch.tensor(0.0, device=device)
        if imitate2.any():
            loss2 = loss2 + dist_weight * criterion(
                out2[imitate2], out1[imitate2].detach())
        if sup.any():
            loss2 = loss2 + criterion(out2[sup], targets[sup])
        loss2.backward()
        optimizer2.step()

        train_loss1 += loss1.item()
        train_loss2 += loss2.item()

    n = len(train_loader)
    return (model1, model2,
            copy.deepcopy(model1.state_dict()),
            copy.deepcopy(model2.state_dict()),
            train_loss1 / n, train_loss2 / n)
    
def dml(model1, model2, optimizer1, optimizer2,
              criterion, distillation_loss_fn, alpha,
              train_loader, device):
    """
    Standard Deep Mutual Learning (DML)
    alpha: Distillation loss의 가중치
    distillation_loss_fn: 모델 간 지식 전달을 위한 함수 (예: KLDiv, MSE)
    """
    model1.train()
    model2.train()
    train_loss1 = 0.0
    train_loss2 = 0.0

    for batch in train_loader:
        inputs = batch["x"].to(device)
        targets = batch["y"].to(device)

        optimizer1.zero_grad()
        optimizer2.zero_grad()

        out1 = model1(inputs)
        out2 = model2(inputs)

        # 1. Supervised Loss (각자 정답(GT)과 비교)
        loss1_sup = criterion(out1, targets)
        loss2_sup = criterion(out2, targets)

        # 2. Distillation Loss (상대방의 예측값과 비교)
        # model1은 model2의 지식을 배우고, model2는 model1의 지식을 배움
        # detach()를 통해 상대방의 그래디언트 흐름을 끊어줌 (Standard DML)
        loss1_dist = distillation_loss_fn(out1, out2.detach())
        loss2_dist = distillation_loss_fn(out2, out1.detach())

        # 3. Total Loss
        loss1 = loss1_sup + alpha * loss1_dist
        loss2 = loss2_sup + alpha * loss2_dist

        loss1.backward()
        loss2.backward()

        optimizer1.step()
        optimizer2.step()

        train_loss1 += loss1.item()
        train_loss2 += loss2.item()

    n = len(train_loader)
    return (model1, model2,
            copy.deepcopy(model1.state_dict()),
            copy.deepcopy(model2.state_dict()),
            train_loss1 / n, train_loss2 / n)
    
# ================================================================
# evaluate — TS 표준 메트릭(MAE + MSE) 추가
# ================================================================
def evaluate(model, test_loader, device):
    model.eval()

    total_mse     = 0.0
    total_mae     = 0.0
    total_rel_l2  = 0.0
    count = 0

    with torch.no_grad():
        for batch in test_loader:
            inputs  = batch["x"].to(device)
            targets = batch["y"].to(device)
            outputs = model(inputs)

            diff = outputs - targets
            B    = targets.size(0)

            # MSE / MAE (TS 논문 표준)
            total_mse += (diff ** 2).mean().item() * B
            total_mae += diff.abs().mean().item()  * B

            # Relative L2 (operator learning과 동일한 척도 — 비교용으로 유지)
            diff_norm   = torch.norm(diff.view(B, -1),    dim=1)
            target_norm = torch.norm(targets.view(B, -1), dim=1)
            total_rel_l2 += (diff_norm / (target_norm + 1e-12)).sum().item()

            count += B

    return {
        "mse":    total_mse   / count,
        "mae":    total_mae   / count,
        "rel_l2": total_rel_l2 / count,
    }

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ETTh1
train_loader, val_loader, test_loader, n_vars = get_ts_loader(
    'ETTm1', batch_size=32, seq_len=336, pred_len=96
)
# → [ETTh1] n_vars=7, train=8001, val=2881, test=2881

# 모델 생성 (이종 쌍)
model1 = build_model('PatchTST', n_vars, seq_len=336, pred_len=96, device=device)
model2 = build_model('TSMixer',  n_vars, seq_len=336, pred_len=96, device=device)


[ETTm1] n_vars=7, train=41377, val=13505, test=13505


print(pretrained_model1)
print(pretrained_model3)
print(pretrained_model4)

In [10]:
def run_all_experiments(seed, dataset_name='ETTh1',
                        seq_len=336, pred_len=96,
                        epochs=50, batch_size=32,
                        n_vars_limit=None):          # ← 추가
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader, val_loader, test_loader, n_vars = get_ts_loader(
        dataset_name, batch_size=batch_size,
        seq_len=seq_len, pred_len=pred_len,
        n_vars_limit=n_vars_limit               # ← 전달
    )

    # ── 초기 모델을 한 번만 만들고 deepcopy로 공유 ──────────────
    set_seed(seed)   # 모델 초기화 seed 고정
    init_pt = build_model('PatchTST', n_vars, seq_len, pred_len, device)
    init_tm = build_model('TSMixer',  n_vars, seq_len, pred_len, device)

    criterion = nn.MSELoss()
    distillation_loss_fn = nn.MSELoss()
    
    # ── Solo 실험 ────────────────────────────────────────────────
    set_seed(seed)
    model_pt = copy.deepcopy(init_pt)
    opt_pt   = torch.optim.Adam(model_pt.parameters(), lr=1e-3, weight_decay=1e-4)
    sch_pt   = torch.optim.lr_scheduler.CosineAnnealingLR(opt_pt, T_max=epochs)
    result_pt = _train_single(model_pt, opt_pt, sch_pt, criterion,
                               train_loader, val_loader, test_loader,
                               epochs, device, f"{dataset_name}/PatchTST/seed{seed}")

    set_seed(seed)
    model_tm = copy.deepcopy(init_tm)
    opt_tm   = torch.optim.Adam(model_tm.parameters(), lr=1e-3, weight_decay=1e-4)
    sch_tm   = torch.optim.lr_scheduler.CosineAnnealingLR(opt_tm, T_max=epochs)
    result_tm = _train_single(model_tm, opt_tm, sch_tm, criterion,
                               train_loader, val_loader, test_loader,
                               epochs, device, f"{dataset_name}/TSMixer/seed{seed}")

    # ── SSML 실험 ────────────────────────────────────────────────
    set_seed(seed)
    m1 = copy.deepcopy(init_pt)   # Solo PatchTST와 동일 초기값
    m2 = copy.deepcopy(init_pt)   # Solo PatchTST와 동일 초기값
    opt1 = torch.optim.Adam(m1.parameters(), lr=1e-3, weight_decay=1e-4)
    opt2 = torch.optim.Adam(m2.parameters(), lr=1e-3, weight_decay=1e-4)
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=epochs)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=epochs)
    result_ssml_pt_pt = _train_ssml(m1, m2, opt1, opt2, sch1, sch2, criterion,
                               train_loader, val_loader, test_loader,
                               epochs, device, f"{dataset_name}/PT+PT_SSML/seed{seed}")


    set_seed(seed)
    m1 = copy.deepcopy(init_tm)   # Solo TSMixer와 동일 초기값
    m2 = copy.deepcopy(init_tm)   # Solo TSMixer와 동일 초기값
    opt1 = torch.optim.Adam(m1.parameters(), lr=1e-3, weight_decay=1e-4)
    opt2 = torch.optim.Adam(m2.parameters(), lr=1e-3, weight_decay=1e-4)
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=epochs)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=epochs)
    result_ssml_tm_tm = _train_ssml(m1, m2, opt1, opt2, sch1, sch2, criterion,
                               train_loader, val_loader, test_loader,
                               epochs, device, f"{dataset_name}/TM+TM_SSML/seed{seed}")
    
    set_seed(seed)
    m1 = copy.deepcopy(init_pt)   # Solo PatchTST와 동일 초기값
    m2 = copy.deepcopy(init_tm)   # Solo TSMixer와 동일 초기값
    opt1 = torch.optim.Adam(m1.parameters(), lr=1e-3, weight_decay=1e-4)
    opt2 = torch.optim.Adam(m2.parameters(), lr=1e-3, weight_decay=1e-4)
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=epochs)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=epochs)
    result_ssml_pt_tm = _train_ssml(m1, m2, opt1, opt2, sch1, sch2, criterion,
                               train_loader, val_loader, test_loader,
                               epochs, device, f"{dataset_name}/PT+TM_SSML/seed{seed}")

    # ── DML 실험 ──────────────────────────────────────────
    set_seed(seed)
    m1, m2 = copy.deepcopy(init_pt), copy.deepcopy(init_pt)
    opt1 = torch.optim.Adam(m1.parameters(), lr=1e-3, weight_decay=1e-4)
    opt2 = torch.optim.Adam(m2.parameters(), lr=1e-3, weight_decay=1e-4)
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=epochs)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=epochs)
    result_dml_pt_pt = _train_dml(m1, m2, opt1, opt2, sch1, sch2, criterion, distillation_loss_fn,
                                   train_loader, val_loader, test_loader,
                                   epochs, device, f"{dataset_name}/PT+TM_DML/seed{seed}")
    
    set_seed(seed)
    m1, m2 = copy.deepcopy(init_tm), copy.deepcopy(init_tm)
    opt1 = torch.optim.Adam(m1.parameters(), lr=1e-3, weight_decay=1e-4)
    opt2 = torch.optim.Adam(m2.parameters(), lr=1e-3, weight_decay=1e-4)
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=epochs)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=epochs)
    result_dml_tm_tm = _train_dml(m1, m2, opt1, opt2, sch1, sch2, criterion, distillation_loss_fn,
                                   train_loader, val_loader, test_loader,
                                   epochs, device, f"{dataset_name}/PT+TM_DML/seed{seed}")

    set_seed(seed)
    m1, m2 = copy.deepcopy(init_pt), copy.deepcopy(init_tm)
    opt1 = torch.optim.Adam(m1.parameters(), lr=1e-3, weight_decay=1e-4)
    opt2 = torch.optim.Adam(m2.parameters(), lr=1e-3, weight_decay=1e-4)
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=epochs)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=epochs)
    result_dml_pt_tm = _train_dml(m1, m2, opt1, opt2, sch1, sch2, criterion, distillation_loss_fn,
                                   train_loader, val_loader, test_loader,
                                   epochs, device, f"{dataset_name}/PT+TM_DML/seed{seed}")

    return {
        'solo_pt':       result_pt,
        'solo_tm':       result_tm,
        'ssml_pt_pt':    result_ssml_pt_pt,
        'ssml_tm_tm':    result_ssml_tm_tm,
        'ssml_pt_tm':    result_ssml_pt_tm,
        'dml_pt_pt':     result_dml_pt_pt,
        'dml_tm_tm':     result_dml_tm_tm,
        'dml_pt_tm':     result_dml_pt_tm,
    }


def _train_single(model, optimizer, scheduler, criterion,
                  train_loader, val_loader, test_loader,
                  epochs, device, desc):
    """Solo 학습 내부 함수"""
    train_curve, val_curve, test_curve = [], [], []
    best_val, best_test, best_state = float("inf"), float("inf"), None

    for epoch in tqdm(range(epochs), desc=desc):
        model, _, train_loss = fine_tune(model, optimizer, criterion,
                                          train_loader, device=device)
        val_m  = evaluate(model, val_loader,  device)
        test_m = evaluate(model, test_loader, device)
        train_curve.append(train_loss)
        val_curve.append(val_m["mse"])
        test_curve.append(test_m["mse"])
        if val_m["mse"] < best_val:
            best_val   = val_m["mse"]
            best_test  = test_m["mse"]
            best_state = copy.deepcopy(model.state_dict())
        scheduler.step()

    return {"best_mse": best_test, "best_val": best_val,
            "train_curve": np.array(train_curve),
            "val_curve":   np.array(val_curve),
            "test_curve":  np.array(test_curve),
            "best_state":  best_state}


def _train_ssml(m1, m2, opt1, opt2, sch1, sch2, criterion,
                train_loader, val_loader, test_loader,
                epochs, device, desc):
    """SSML 학습 내부 함수"""
    train_curve1, val_curve1, test_curve1 = [], [], []
    train_curve2, val_curve2, test_curve2 = [], [], []
    best_val1, best_test1, best_state1 = float("inf"), float("inf"), None
    best_val2, best_test2, best_state2 = float("inf"), float("inf"), None

    for epoch in tqdm(range(epochs), desc=desc):
        # Warm-up: 5에폭까지는 일반 Fine-tune, 이후 SSML 적용
        if epoch < 5:
            m1, _, tl1 = fine_tune(m1, opt1, criterion, train_loader, device)
            m2, _, tl2 = fine_tune(m2, opt2, criterion, train_loader, device)
        else:
            m1, m2, _, _, tl1, tl2 = ssml(m1, m2, opt1, opt2, criterion, train_loader, device)
        val1  = evaluate(m1, val_loader,  device)
        val2  = evaluate(m2, val_loader,  device)
        test1 = evaluate(m1, test_loader, device)
        test2 = evaluate(m2, test_loader, device)

        train_curve1.append(tl1); val_curve1.append(val1["mse"]); test_curve1.append(test1["mse"])
        train_curve2.append(tl2); val_curve2.append(val2["mse"]); test_curve2.append(test2["mse"])

        if val1["mse"] < best_val1:
            best_val1 = val1["mse"]; best_test1 = test1["mse"]
            best_state1 = copy.deepcopy(m1.state_dict())
        if val2["mse"] < best_val2:
            best_val2 = val2["mse"]; best_test2 = test2["mse"]
            best_state2 = copy.deepcopy(m2.state_dict())
        sch1.step(); sch2.step()

    return {"best_mse1": best_test1, "best_val1": best_val1,
            "train_curve1": np.array(train_curve1), "val_curve1": np.array(val_curve1),
            "test_curve1":  np.array(test_curve1),  "best_state1": best_state1,
            "best_mse2": best_test2, "best_val2": best_val2,
            "train_curve2": np.array(train_curve2), "val_curve2": np.array(val_curve2),
            "test_curve2":  np.array(test_curve2),  "best_state2": best_state2}

def _train_dml(m1, m2, opt1, opt2, sch1, sch2, criterion, distillation_loss_fn,
               train_loader, val_loader, test_loader,
               epochs, device, desc):
    """DML 학습 내부 함수 — _train_ssml과 동일 구조, dml() 함수만 다름"""
    train_curve1, val_curve1, test_curve1 = [], [], []
    train_curve2, val_curve2, test_curve2 = [], [], []
    best_val1, best_test1, best_state1 = float("inf"), float("inf"), None
    best_val2, best_test2, best_state2 = float("inf"), float("inf"), None

    for epoch in tqdm(range(epochs), desc=desc):
        # Warm-up: 5에폭까지는 일반 Fine-tune, 이후 SSML 적용
        if epoch < 5:
            m1, _, tl1 = fine_tune(m1, opt1, criterion, train_loader, device)
            m2, _, tl2 = fine_tune(m2, opt2, criterion, train_loader, device)
        else:
            m1, m2, _, _, tl1, tl2 = dml(m1, m2, opt1, opt2,   # ← ssml → dml
                                       criterion, distillation_loss_fn, 0.5, train_loader, device=device)
        val1  = evaluate(m1, val_loader,  device)
        val2  = evaluate(m2, val_loader,  device)
        test1 = evaluate(m1, test_loader, device)
        test2 = evaluate(m2, test_loader, device)

        train_curve1.append(tl1); val_curve1.append(val1["mse"]); test_curve1.append(test1["mse"])
        train_curve2.append(tl2); val_curve2.append(val2["mse"]); test_curve2.append(test2["mse"])

        if val1["mse"] < best_val1:
            best_val1 = val1["mse"]; best_test1 = test1["mse"]
            best_state1 = copy.deepcopy(m1.state_dict())
        if val2["mse"] < best_val2:
            best_val2 = val2["mse"]; best_test2 = test2["mse"]
            best_state2 = copy.deepcopy(m2.state_dict())
        sch1.step(); sch2.step()

    return {"best_mse1": best_test1, "best_val1": best_val1,
            "train_curve1": np.array(train_curve1), "val_curve1": np.array(val_curve1),
            "test_curve1":  np.array(test_curve1),  "best_state1": best_state1,
            "best_mse2": best_test2, "best_val2": best_val2,
            "train_curve2": np.array(train_curve2), "val_curve2": np.array(val_curve2),
            "test_curve2":  np.array(test_curve2),  "best_state2": best_state2}

# ================================================================
# 실행
# ================================================================
seeds = [0, 1, 2, 3, 4]
all_results = []

for seed in tqdm(seeds, desc="seeds"):
    all_results.append(
        run_all_experiments(seed, dataset_name='ETTh1', epochs=100, batch_size=64))



seeds:   0%|          | 0/5 [00:00<?, ?it/s]

[ETTh1] n_vars=7, train=10021, val=3053, test=3053


ETTh1/PatchTST/seed0:   0%|          | 0/100 [00:00<?, ?it/s]

ETTh1/TSMixer/seed0:   0%|          | 0/100 [00:00<?, ?it/s]

ETTh1/PT+PT_SSML/seed0:   0%|          | 0/100 [00:00<?, ?it/s]

...

ETTh1/TM+TM_SSML/seed4:   0%|          | 0/100 [00:00<?, ?it/s]

ETTh1/PT+TM_SSML/seed4:   0%|          | 0/100 [00:00<?, ?it/s]

ETTh1/PT+TM_DML/seed4:   0%|          | 0/100 [00:00<?, ?it/s]

ETTh1/PT+TM_DML/seed4:   0%|          | 0/100 [00:00<?, ?it/s]

ETTh1/PT+TM_DML/seed4:   0%|          | 0/100 [00:00<?, ?it/s]

In [11]:
# 결과 추출
results_PatchTST = [r['solo_pt']    for r in all_results]
results_TSMixer  = [r['solo_tm']    for r in all_results]
results_PT_PT    = [r['ssml_pt_pt'] for r in all_results]
results_TM_TM    = [r['ssml_tm_tm'] for r in all_results]
results_PT_TM    = [r['ssml_pt_tm'] for r in all_results]
results_PT_PT_dml    = [r['dml_pt_pt'] for r in all_results]
results_TM_TM_dml    = [r['dml_tm_tm'] for r in all_results]
results_PT_TM_dml    = [r['dml_pt_tm'] for r in all_results]

# 비교 출력
print(f"PatchTST solo : {np.mean([r['best_mse']  for r in results_PatchTST]):.4f}")
print(f"TSMixer  solo : {np.mean([r['best_mse']  for r in results_TSMixer]):.4f}")
print(f"PT+PT SSML PT : {np.mean([r['best_mse1'] for r in results_PT_PT]):.4f}")
print(f"PT+Pt SSML PT : {np.mean([r['best_mse2'] for r in results_PT_TM]):.4f}")
print(f"TM+TM SSML TM : {np.mean([r['best_mse1'] for r in results_TM_TM]):.4f}")
print(f"TM+TM SSML TM : {np.mean([r['best_mse2'] for r in results_TM_TM]):.4f}")
print(f"PT+TM SSML PT : {np.mean([r['best_mse1'] for r in results_PT_TM]):.4f}")
print(f"PT+TM SSML TM : {np.mean([r['best_mse2'] for r in results_PT_TM]):.4f}")
print(f"PT+PT DML PT : {np.mean([r['best_mse1'] for r in results_PT_PT_dml]):.4f}")
print(f"PT+PT DML PT : {np.mean([r['best_mse2'] for r in results_PT_PT_dml]):.4f}")
print(f"TM+TM DML TM : {np.mean([r['best_mse1'] for r in results_TM_TM_dml]):.4f}")
print(f"TM+TM DML TM : {np.mean([r['best_mse2'] for r in results_TM_TM_dml]):.4f}")
print(f"PT+TM DML PT : {np.mean([r['best_mse1'] for r in results_PT_TM_dml]):.4f}")
print(f"PT+TM DML TM : {np.mean([r['best_mse2'] for r in results_PT_TM_dml]):.4f}")

PatchTST solo : 0.4432
TSMixer  solo : 0.5273
PT+PT SSML PT : 0.4384
PT+Pt SSML PT : 0.4853
TM+TM SSML TM : 0.4969
TM+TM SSML TM : 0.5017
PT+TM SSML PT : 0.4449
PT+TM SSML TM : 0.4853
PT+PT DML PT : 0.4384
PT+PT DML PT : 0.4469
TM+TM DML TM : 0.4827
TM+TM DML TM : 0.4835
PT+TM DML PT : 0.4426
PT+TM DML TM : 0.4954


In [12]:
import csv
import os

os.makedirs('results/ts', exist_ok=True)

# ================================================================
# 헬퍼 함수 — 반복 코드 제거
# ================================================================
def save_single(results, filepath):
    """run_experiment_ts 결과 저장"""
    fieldnames = ['best_mse', 'best_val',
                  'train_curve', 'val_curve', 'test_curve']
    with open(filepath, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in results:
            writer.writerow({
                'best_mse':    r['best_mse'],
                'best_val':    r['best_val'],
                'train_curve': r['train_curve'].tolist(),
                'val_curve':   r['val_curve'].tolist(),
                'test_curve':  r['test_curve'].tolist(),
            })

def save_ssml(results, filepath):
    """run_experiment_ssml_ts 결과 저장"""
    fieldnames = ['best_mse1', 'best_val1',
                  'train_curve1', 'val_curve1', 'test_curve1',
                  'best_mse2', 'best_val2',
                  'train_curve2', 'val_curve2', 'test_curve2']
    with open(filepath, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in results:
            writer.writerow({
                'best_mse1':   r['best_mse1'],
                'best_val1':   r['best_val1'],
                'train_curve1': r['train_curve1'].tolist(),
                'val_curve1':   r['val_curve1'].tolist(),
                'test_curve1':  r['test_curve1'].tolist(),
                'best_mse2':   r['best_mse2'],
                'best_val2':   r['best_val2'],
                'train_curve2': r['train_curve2'].tolist(),
                'val_curve2':   r['val_curve2'].tolist(),
                'test_curve2':  r['test_curve2'].tolist(),
            })

BASE = 'results/ts/ETTh1'
os.makedirs(BASE, exist_ok=True)

save_single(results_PatchTST,      f'{BASE}/patchtst100.csv')
save_single(results_TSMixer,      f'{BASE}/tsmixer100.csv')
save_ssml(results_PT_PT,   f'{BASE}/patchtst_patchtst_ssml100.csv')
save_ssml(results_TM_TM,   f'{BASE}/tsmixer_tsmixer_ssml100.csv')
save_ssml(results_PT_TM,   f'{BASE}/patchtst_tsmixer_ssml100.csv')
save_ssml(results_PT_PT_dml,    f'{BASE}/patchtst_patchtst_dml100.csv')
save_ssml(results_TM_TM_dml,    f'{BASE}/tsmixer_tsmixer_dml100.csv')
save_ssml(results_PT_TM_dml,    f'{BASE}/patchtst_tsmixer_dml100.csv')
print("저장 완료")

저장 완료
